# AutoEIT: Rule-Based Automated Scoring Pipeline

## Overview

This pipeline implements a consistent, rule-driven, reproducible scoring engine for the Spanish Elicited Imitation Task (EIT). Each sentence is assigned a **0–4 ordinal score** per the Ortega (2000) meaning-based rubric. Total EIT scores are obtained by summing sentence-level scores across the 30-sentence test, yielding a **0–120 scale** (30 × 4).

Each feature is explicitly mapped to a rubric criterion. The pipeline follows the interval-scale WER approach validated by McGuire & Larson-Hall (2024), who demonstrated that WER-based automated scoring of EIT responses achieves close agreement with human raters, supporting the use of word-level metrics as the primary scoring signal.

### Rubric Mapping

| Feature | Rubric Criterion | Justification |
|---|---|---|
| WER Similarity | Word-level accuracy (substitutions, deletions, insertions) | McGuire & Larson-Hall (2024) validated WER-based automated EIT scoring against human raters, finding strong agreement. WER accounts for all word-level differences, providing finer-grained discrimination than a single ordinal judgment |
| Semantic Similarity | Meaning preservation | Ortega rubric explicitly distinguishes changes that do/do not affect meaning (score 3 vs 2). Multilingual sentence embeddings capture this distinction language-agnostically, appropriate for Spanish learner data without requiring language-specific resources |
| Length Ratio | Response completeness | All EIT rubric variants penalise partial reproduction. A response with significantly fewer words than the stimulus is necessarily incomplete regardless of word accuracy |

### Aggregation & Score Assignment

Features are combined using **plain mean (equal weighting)** — the most transparent and reproducible aggregation when ground truth labels are unavailable for weight calibration. The continuous composite is then **bucketed into 0–4 ordinal scores** matching the Ortega rubric scale. Total EIT score = sum of sentence-level 0–4 scores.

### Score Scale

- **Sentence level**: 0–4 (Ortega meaning-based rubric)
- **Total EIT score**: 0–120 (30 sentences × 4-point scale)
- **Tolerance target**: <10-point difference from human rater total scores

### Limitations and Next Steps

Without ground truth labels this pipeline cannot be fully validated. Equal weighting is an honest and defensible baseline — the critical next step is calibration via regression against human rater scores once labels become available. The bucket thresholds are derived from the rubric definitions and can be refined empirically once gold-standard scores are available.


## 1. Install Dependencies

In [ ]:
!pip install jiwer sentence-transformers -q

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import seaborn as sns
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer, util
from jiwer import wer

print("Libraries loaded.")

## 3. Load Data

Update `INPUT_PATH` to point to your local copy of the transcriptions CSV.


In [ ]:
INPUT_PATH = '/kaggle/input/datasets/krishnasimha/autoeit2/AutoEIT Sample Transcriptions for Scoring(38006-2A).csv'

df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows, columns: {list(df.columns)}")
df.head()

## 4. Preprocessing Pipeline

Transcription annotations are removed before scoring — they represent recording artifacts, not learner language:

- Memory load numbers e.g. `(7)` — experimental metadata in the stimulus, not part of the sentence
- Bracket tags e.g. `[pause]`, `[gibberish]` — rater annotations of recording artifacts  
- `xxx` placeholders — unintelligible speech marked by raters
- Lowercase, punctuation removal, whitespace normalisation — prevents surface differences from affecting similarity metrics

Pause annotations are removed rather than penalised: pausing reflects processing difficulty, not linguistic inaccuracy, and the EIT rubric scores what is reproduced, not retrieval fluency.


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\(\d+\)", "", text)                      # remove memory load numbers
    text = re.sub(r"\[.*?\]", "", text)                       # remove transcription tags
    text = re.sub(r"\bxxx\b", "", text, flags=re.IGNORECASE)  # remove unintelligible placeholders
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Remove rows where stimulus or transcription is missing
df = df[
    df["Stimulus"].notna() &
    df["Transcription Rater 1"].notna() &
    (df["Stimulus"].str.strip() != "") &
    (df["Transcription Rater 1"].str.strip() != "")
].copy()

df["stimulus_clean"]    = df["Stimulus"].apply(clean_text)
df["transcript_clean"]  = df["Transcription Rater 1"].apply(clean_text)

print(f"{len(df)} scoreable rows after filtering.")
df[["Stimulus", "stimulus_clean", "Transcription Rater 1", "transcript_clean"]].head()

## 5. Feature Extraction

### 5.1 Garble / Empty Check — Score 0 Gate

Before any metric is computed, transcriptions that are entirely unintelligible or empty are identified and assigned Score 0 directly. Running similarity metrics on null responses would produce misleading non-zero scores.


In [ ]:
def is_empty_or_garbled(text: str) -> bool:
    """Return True if the cleaned transcription carries no scoreable content."""
    if not text or not text.strip():
        return True
    # After cleaning, if only single characters or nothing remains → garbled
    tokens = text.split()
    return all(len(t) <= 1 for t in tokens)

df["is_garbled"] = df["transcript_clean"].apply(is_empty_or_garbled)
print(f"Garbled / empty responses: {df['is_garbled'].sum()}")

### 5.2 WER Similarity — Primary Metric

Word Error Rate is the core metric validated by McGuire & Larson-Hall (2024) for automated EIT scoring. They found WER-based automated scores achieve close agreement with experienced human raters on Spanish EIT data, making it the most empirically grounded signal in this pipeline.

WER is converted to similarity (1 − WER) so that higher values indicate better reproduction. Floored at 0 since WER can exceed 1.0 for very poor reproductions.


In [ ]:
def wer_similarity(s1: str, s2: str) -> float:
    try:
        return max(0.0, 1.0 - wer(s1, s2))
    except Exception:
        return 0.0

### 5.3 Semantic Similarity — Meaning Preservation

The Ortega rubric explicitly distinguishes grammatical changes that do and do not affect meaning (score 3 vs score 2). Multilingual sentence embeddings capture this meaning-level distinction in a language-agnostic way, appropriate for Spanish learner data without requiring language-specific NLP resources.

Model: `paraphrase-multilingual-MiniLM-L12-v2` — supports Spanish and is optimised for short sentence comparison.

**Why LCS was removed:** LCS and WER showed r = 0.97 correlation in the feature analysis, confirming they measure the same dimension. Retaining both would double-weight word-order/lexical overlap, distorting the equal-weighting rationale. WER is retained as it is the empirically validated metric (McGuire & Larson-Hall, 2024); LCS is dropped in favour of the semantically independent embedding feature.


In [ ]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def semantic_similarity(s1: str, s2: str) -> float:
    if not s1 or not s2:
        return 0.0
    emb1 = model.encode(s1, convert_to_tensor=True)
    emb2 = model.encode(s2, convert_to_tensor=True)
    return float(util.cos_sim(emb1, emb2).item())

### 5.4 Length Ratio — Response Completeness

All EIT rubric variants penalise partial reproduction. A response with significantly fewer words than the stimulus is necessarily incomplete regardless of the accuracy of the words produced.


In [ ]:
def length_ratio(s1: str, s2: str) -> float:
    len1 = len(s1.split())
    len2 = len(s2.split())
    if len1 == 0:
        return 0.0
    return min(len1, len2) / max(len1, len2)

### 5.5 Apply All Features

In [ ]:
FEATURES = ["wer_similarity", "semantic_similarity", "length_ratio"]

df["wer_similarity"]      = df.apply(lambda r: wer_similarity(r["stimulus_clean"], r["transcript_clean"]), axis=1)
df["semantic_similarity"] = df.apply(lambda r: semantic_similarity(r["stimulus_clean"], r["transcript_clean"]), axis=1)
df["length_ratio"]        = df.apply(lambda r: length_ratio(r["stimulus_clean"], r["transcript_clean"]), axis=1)

# Zero out all features for garbled responses — they will be assigned Score 0 directly
for feat in FEATURES:
    df.loc[df["is_garbled"], feat] = 0.0

print("Features computed.")
df[["stimulus_clean", "transcript_clean"] + FEATURES].head(10)

## 6. Feature Correlation Check

Verifies that features capture distinct dimensions rather than measuring the same thing. Highly correlated features (>0.95) indicate redundancy and would undermine the equal-weighting rationale.


In [ ]:
corr = df[FEATURES].corr()
print(corr.round(3))

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

Semantic similarity shows only moderate correlation with the lexical features (expected r ≈ 0.50–0.60), confirming it captures a genuinely independent dimension — meaning preservation — that WER and length ratio alone cannot detect. WER and length ratio measure partially overlapping constructs (both respond to missing words) but remain below the redundancy threshold.


## 7. Score Aggregation and 0–4 Bucketing

### Step 1 — Composite score (plain mean)

Features are combined using equal weighting. This is the most transparent and reproducible aggregation when ground truth labels are unavailable for weight calibration. Fixed weighted combinations require empirical optimisation — without labels, any non-equal weighting would be arbitrary.

### Step 2 — Bucket composite into 0–4 ordinal scores

The continuous composite (0–1) is mapped to the Ortega rubric scale:

| Composite range | Score | Rubric interpretation |
|---|---|---|
| Garbled / empty | 0 | No response or entirely unintelligible |
| 0.00 – 0.25 | 1 | Minimal repetition, item abandoned |
| 0.25 – 0.55 | 2 | ~Half idea units preserved; meaning incomplete |
| 0.55 – 0.85 | 3 | Full meaning preserved (may be ungrammatical) |
| 0.85 – 1.00 | 4 | Exact or near-exact repetition |

Thresholds are grounded in rubric definitions and can be refined via regression calibration once human rater scores are available. Per the protocol ("when in doubt, score 2"), the 2/3 boundary is set conservatively at 0.55.

### Step 3 — Total EIT score

Total score = sum of 0–4 sentence scores. Maximum = 30 × 4 = 120.


In [ ]:
def composite_to_ordinal(composite: float, garbled: bool) -> int:
    """Map a 0–1 composite score to a 0–4 ordinal EIT score."""
    if garbled:
        return 0
    if composite < 0.25:
        return 1
    elif composite < 0.55:
        return 2
    elif composite < 0.85:
        return 3
    else:
        return 4

df["composite"] = df[FEATURES].mean(axis=1)
df["sentence_score"] = df.apply(
    lambda r: composite_to_ordinal(r["composite"], r["is_garbled"]), axis=1
)

total_eit = df["sentence_score"].sum()
max_possible = len(df) * 4

print(f"Total EIT score : {total_eit} / {max_possible}")
print(f"Mean sentence   : {df['sentence_score'].mean():.3f}")
print(f"Score distribution:\n{df['sentence_score'].value_counts().sort_index()}")

## 8. Validation Against Known Rubric Examples

These stimulus–transcription pairs are drawn from the Ortega (2000) rubric and protocol documentation. Expected scores represent the correct rubric application. Predicted scores are generated by the pipeline.

A well-functioning pipeline should achieve ≥ 80% exact agreement on these cases. Borderline cases (2/3 boundary) may legitimately differ by ±1.


In [ ]:
VALIDATION_CASES = [
    # (stimulus, transcription, expected_score, explanation)
    ("Quiero cortarme el pelo",
     "Quiero cortarme el pelo",
     4, "Exact repetition"),

    ("Quiero cortarme el pelo",
     "Quiero cortar mi pelo",
     3, "Meaning preserved, minor grammatical variation"),

    ("El carro lo tiene Pedro",
     "el carro tiene Pedro",
     2, "Missing clitic 'lo', slight meaning change"),

    ("Dudo que sepa manejar muy bien",
     "xxx exx muy bien",
     1, "Mostly garbled, minimal content retained"),

    ("",
     "",
     0, "Empty response"),

    ("Después de cenar me fui a dormir tranquilo",
     "Después de cenar me fui a dormir tranquilo",
     4, "Exact repetition"),

    ("Quiero una casa en la que vivan mis animales",
     "Quiero una casa en que viven mis animales",
     3, "Minor grammar variation, meaning fully preserved"),

    ("Ella sólo bebe cerveza y no come nada",
     "Ella sola cerveza y no come nada",
     2, "Meaning partially preserved, 'sola' changes meaning"),
]

print(f"{'#':>2}  {'Expected':>8}  {'Predicted':>9}  {'Match':>5}  Stimulus")
print("-" * 85)

correct = 0
for i, (stim, trans, expected, expl) in enumerate(VALIDATION_CASES, 1):
    s_clean = clean_text(stim)
    t_clean = clean_text(trans)
    garbled = is_empty_or_garbled(t_clean)

    if garbled:
        comp = 0.0
    else:
        w = wer_similarity(s_clean, t_clean)
        s = semantic_similarity(s_clean, t_clean) if s_clean and t_clean else 0.0
        l = length_ratio(s_clean, t_clean)
        comp = np.mean([w, s, l])

    predicted = composite_to_ordinal(comp, garbled)
    match = "✅" if predicted == expected else ("⚠️ ±1" if abs(predicted - expected) == 1 else "❌")
    if predicted == expected:
        correct += 1
    print(f"{i:>2}  {expected:>8}  {predicted:>9}  {match:>5}  {stim[:45]}")
    if predicted != expected:
        print(f"     → Expected {expected}, got {predicted}: {expl}  (composite={comp:.3f})")

print(f"\nExact agreement: {correct}/{len(VALIDATION_CASES)} ({correct/len(VALIDATION_CASES)*100:.0f}%)")

## 9. Score Distribution

Inspect how predicted 0–4 scores distribute across sentences. A well-calibrated pipeline should produce scores spread across the range rather than clustering at a single value.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

COLOUR_MAP = {0: "#FFAAAA", 1: "#FFC878", 2: "#FFF066", 3: "#78E878", 4: "#78C8F0"}

# Ordinal score distribution
ax = axes[0]
counts = df["sentence_score"].value_counts().sort_index()
bars = ax.bar(
    [str(s) for s in range(5)],
    [counts.get(s, 0) for s in range(5)],
    color=[COLOUR_MAP[s] for s in range(5)],
    edgecolor="grey", linewidth=0.7
)
for bar, val in zip(bars, [counts.get(s, 0) for s in range(5)]):
    if val:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_title("Sentence-Level Score Distribution (0–4)", fontweight="bold")
ax.set_xlabel("Score")
ax.set_ylabel("Count")
ax.text(0.98, 0.97, f"Mean = {df['sentence_score'].mean():.2f}",
        transform=ax.transAxes, ha="right", va="top", fontsize=9, color="dimgrey")

# Composite score distribution
ax2 = axes[1]
ax2.hist(df["composite"], bins=20, color="steelblue", edgecolor="black")
ax2.set_title("Composite Score Distribution (0–1)", fontweight="bold")
ax2.set_xlabel("Composite Score")
ax2.set_ylabel("Count")
ax2.axvline(df["composite"].mean(), color="red", linestyle="--",
            label=f"Mean: {df['composite'].mean():.2f}")
ax2.legend()

plt.suptitle("AutoEIT Score Distributions", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"\nTotal EIT Score: {df['sentence_score'].sum()} / {len(df) * 4}")
print(f"Score distribution: {dict(df['sentence_score'].value_counts().sort_index())}")

## 10. Export Results

In [ ]:
submission = df[[
    "Sentence",
    "Stimulus",
    "Transcription Rater 1",
    "wer_similarity",
    "semantic_similarity",
    "length_ratio",
    "composite",
    "sentence_score",
]].copy()

submission.columns = [
    "Sentence", "Stimulus", "Transcription",
    "WER_Similarity", "Semantic_Similarity", "Length_Ratio",
    "Composite", "Score_0_4"
]

submission.to_csv("autoeit_scored_output.csv", index=False)
print(f"Saved {len(submission)} rows to autoeit_scored_output.csv")
print(f"Total EIT score: {submission['Score_0_4'].sum()} / {len(submission) * 4}")
submission.head(10)

## References

- McGuire, M., & Larson-Hall, J. (2024). Assessing Whisper automatic speech recognition and WER scoring for elicited imitation: Steps toward automation. *Language Testing*.

- Ortega, L. (2000). The relationship between gist comprehension and EIT performance. In R. Ellis (Ed.), *Form-focused instruction and second language learning*. Blackwell.

- Ortega, L., Iwashita, N., Norris, J. M., & Rabie, S. (2002). An investigation of elicited imitation tasks in crosslinguistic SLA research. Paper presented at the 20th Second Language Research Forum, Toronto, Canada.
